# Tile QC — EMIT Cloud Mask + Reverse R²

Runs after `Color_Matching.ipynb`. Downloads EMIT L2A mask NCs, reprojects
cloud/cirrus flags to the same UTM grid, computes EMIT→S2 reverse R² to detect
S2 cloud contamination. All thresholds come from `pipeline_config.yaml`.

In [ ]:
import os, textwrap
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

In [ ]:
!apt-get install -qq gdal-bin libgdal-dev > /dev/null
!pip install -q numpy scipy rasterio matplotlib tqdm earthaccess netCDF4 pyproj shapely xarray python-dateutil pyyaml scikit-learn

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import earthaccess as ea
ea.login(strategy="environment")

In [ ]:
DRIVE_BASE = "/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22"

!python data/tile_qc.py --drive-base "$DRIVE_BASE"

## Diagnostics — distributions & threshold sweep

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

drive_base = Path(DRIVE_BASE)
with open(drive_base / "pipeline_config.yaml") as f:
    cfg = yaml.safe_load(f)

max_cloud = cfg["qc_max_emit_cloud_frac"]
min_rev = cfg["qc_min_r2_reverse"]

qc = pd.read_csv(drive_base / "r2_tiles_qc.csv")
print(f"Total tiles: {len(qc)}")

# ── 1. Marginal distributions ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(qc["combined_frac"].dropna(), bins=60, edgecolor="black", alpha=0.7)
axes[0].axvline(max_cloud, color="red", ls="--", label=f"thresh={max_cloud}")
axes[0].set(xlabel="EMIT cloud+cirrus frac", ylabel="Count")
axes[0].legend()

axes[1].hist(qc["r2_reverse"].dropna(), bins=60, edgecolor="black", alpha=0.7)
axes[1].axvline(min_rev, color="red", ls="--", label=f"thresh={min_rev}")
axes[1].set(xlabel="Reverse R² (EMIT→S2)", ylabel="Count")
axes[1].legend()

axes[2].hist(qc["r2_mean"].dropna(), bins=60, edgecolor="black", alpha=0.7)
axes[2].set(xlabel="Forward R² (S2→EMIT)", ylabel="Count")

plt.tight_layout()
plt.show()

# ── 2. Forward vs reverse R² scatter ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(qc["r2_mean"], qc["r2_reverse"], alpha=0.15, s=4, c="steelblue")
axes[0].axhline(min_rev, color="red", ls="--", lw=0.8)
axes[0].plot([0, 1], [0, 1], "k--", lw=0.5, alpha=0.3)
axes[0].set(xlabel="Forward R² (S2→EMIT)", ylabel="Reverse R² (EMIT→S2)")

axes[1].scatter(qc["r2_mean"], qc["combined_frac"], alpha=0.15, s=4, c="darkorange")
axes[1].axhline(max_cloud, color="red", ls="--", lw=0.8)
axes[1].set(xlabel="Forward R²", ylabel="Cloud+cirrus frac")

axes[2].scatter(qc["r2_reverse"], qc["combined_frac"], alpha=0.15, s=4, c="seagreen")
axes[2].axhline(max_cloud, color="red", ls="--", lw=0.8)
axes[2].axvline(min_rev, color="red", ls="--", lw=0.8)
axes[2].set(xlabel="Reverse R²", ylabel="Cloud+cirrus frac")

plt.tight_layout()
plt.show()

# ── 3. R² gap: tiles where forward >> reverse (S2 cloud suspects) ────
qc["r2_gap"] = qc["r2_mean"] - qc["r2_reverse"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(qc["r2_gap"].dropna(), bins=80, edgecolor="black", alpha=0.7)
ax.axvline(0, color="gray", ls="-", lw=0.5)
ax.set(xlabel="R² gap (forward − reverse)", ylabel="Count",
       title="Positive gap = suspect S2 cloud contamination")
plt.tight_layout()
plt.show()

# ── 4. Threshold sweep (R² × cloud × reverse R²) ────────────────────
r2_cuts = [0.0, 0.65, 0.70, 0.75, 0.80, 0.85]
cloud_cuts = [1.0, 0.10, 0.05, 0.02, 0.0]
rev_cuts = [0.0, 0.30, 0.40, 0.50, 0.60, 0.70]

print("\n── Threshold sweep (R² × cloud × reverse R²) ──")
print(f"{'R²≥':>6} {'cloud≤':>8} {'rev≥':>7} {'pass':>7} {'%':>7}")
print("-" * 40)
for r2t in r2_cuts:
    for ct in cloud_cuts:
        for rt in rev_cuts:
            mask = (qc["r2_mean"] >= r2t) & (qc["combined_frac"] <= ct) & (qc["r2_reverse"] >= rt)
            n = mask.sum()
            if n == 0:
                continue
            print(f"{r2t:>6.2f} {ct:>8.2f} {rt:>7.2f} {n:>7d} {100*n/len(qc):>6.1f}%")

# ── 5. Key intersections ─────────────────────────────────────────────
print("\n── Key intersections ──")
for r2t in [0.70, 0.75, 0.80]:
    r2_ok = qc["r2_mean"] >= r2t
    cloud_ok = qc["combined_frac"] <= max_cloud
    rev_ok = qc["r2_reverse"] >= min_rev

    n_r2 = r2_ok.sum()
    n_cloud = (r2_ok & cloud_ok).sum()
    n_rev = (r2_ok & rev_ok).sum()
    n_all = (r2_ok & cloud_ok & rev_ok).sum()

    print(f"\nR² >= {r2t}:")
    print(f"  R² only:              {n_r2:>6d}")
    print(f"  + cloud ≤ {max_cloud}:       {n_cloud:>6d}  (−{n_r2-n_cloud})")
    print(f"  + reverse ≥ {min_rev}:     {n_rev:>6d}  (−{n_r2-n_rev})")
    print(f"  all three:            {n_all:>6d}  (−{n_r2-n_all} from R² set)")

## Tile examples — pass vs fail categories

In [ ]:
import sys
sys.path.insert(0, "/content/HyperSpectralSuperResolution")
from data.tile_qc import plot_tile_examples

plot_tile_examples(qc, n=5, out_path=str(drive_base / "qc_tile_examples.png"))